<a href="https://colab.research.google.com/github/Lauri06/Algoritmo-KNN---Machine-Learning/blob/main/Taller_5_ML_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificador KNN con Embeddings de Texto

**Dataset:** E-commerce / Call Center

**Integrantes:**
- Laura Camila Gutiérrez Hernández
- Cristian Camilo Posada García

## Librerias y Carga del Dataset

In [ ]:
import csv
import random
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

In [ ]:
# Lee el CSV y retorna listas de textos y clases
textos = []
clases = []
ruta_csv = 'dataset_ecommerce_callcenter.csv'
with open(ruta_csv, newline='', encoding='utf-8') as archivo:
  lector = csv.DictReader(archivo)
  for fila in lector:
    textos.append(fila['texto'].strip())
    clases.append(fila['clase'].strip())

clases_unicas = sorted(set(clases))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
textos

In [ ]:
clases

In [ ]:
clases_unicas

## Generar Embeddings

In [ ]:
def generar_embeddings(modelo, textos):
  """Convierte una lista de textos en vectores numéricos (embeddings)"""
  embeddings = modelo.encode(textos, show_progress_bar=True)
  return [list(map(float, v)) for v in embeddings]

In [ ]:
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = generar_embeddings(modelo, textos)

## Metricas de Similitud

### Similitud Coseno $$\cos{(\theta)} = \frac{v_1 \cdot v_2}{||v_1||\cdot ||v_2||}$$

In [ ]:
def similitud_coseno(v1, v2):
  """
  Similitud del coseno entre dos vectores
  Devuelve un valor en [-1, 1]; 1 = identicos
  """
  producto = sum(a * b for a, b in zip(v1, v2))
  norma1 = sum(a ** 2 for a in v1) ** 0.5
  norma2 = sum(b ** 2 for b in v2) ** 0.5
  if norma1 == 0 or norma2 == 0:
      return 1.0 # Maxima distancia si alguno es nulo
  similitud = producto / (norma1 * norma2)
  return 1.0 - similitud # Convertir a distancia (menor = más cercano)

### Distancia Manhattan $$L_1 = \sum{|v1_i-v2_i|}$$

In [ ]:
def distancia_l1(v1, v2):
  """
  Rango: [0, infinito) Cuanto menor, mas similares
  """
  return sum(abs(a - b) for a, b in zip(v1, v2))

### Distancia Euclidiana $$L_2 = \sqrt{\sum{(v1_i-v2_i)}^2}$$

In [ ]:
def distancia_l2(v1, v2):
  """
  Rango: [0, infinito) Cuanto menor, mas similares
  """
  return sum((a - b) ** 2 for a, b in zip(v1, v2)) ** 0.5

### Distancia Minkowski $$L_p = (\sum{|v1_i-v2_i|})^\frac{1}{p}$$

In [ ]:
def distancia_minkowski(v1, v2, p=3):
  """
  Generalizacion: p=1 -> Manhattan, p=2 -> Euclidiana
  Se usa p=3 como norma elegida
  """
  return sum(abs(a - b) ** p for a, b in zip(v1, v2)) ** (1 / p)

### Diccionario de Metricas

In [ ]:
metricas = {
    '1': ('Coseno',     similitud_coseno),
    '2': ('Manhattan',  distancia_l1),
    '3': ('Euclidiana', distancia_l2),
    '4': ('Minkowski',  distancia_minkowski),
}

## Algoritmo KNN

In [ ]:
def calcular_distancias(embedding_consulta, embeddings_train, fn_distancia):
  """
  Calcula la distancia entre la consulta y cada ejemplo del conjunto
  """
  distancias = []
  for i, emb in enumerate(embeddings_train):
      d = fn_distancia(embedding_consulta, emb)
      distancias.append((i, d))
  return distancias


def seleccionar_k_vecinos(distancias, k):
  """
  Ordena por distancia (ascendente) y retorna los k más cercanos
  """
  ordenadas = sorted(distancias, key=lambda x: x[1])
  return ordenadas[:k]


def votar_clase(vecinos, clases_train):
  """
  Decisión por mayoría de votos entre los k vecinos
  """
  conteo = {}
  for idx, _ in vecinos:
      clase = clases_train[idx]
      conteo[clase] = conteo.get(clase, 0) + 1
  # Clase con mas votos
  return max(conteo, key=lambda c: conteo[c]), conteo


def clasificar(embedding_consulta, embeddings_train, clases_train, k, fn_distancia):
  """
  Flujo completo de KNN para un unico punto de consulta
  """
  distancias = calcular_distancias(embedding_consulta, embeddings_train, fn_distancia)
  vecinos = seleccionar_k_vecinos(distancias, k)
  clase_pred, conteo = votar_clase(vecinos, clases_train)
  return clase_pred, vecinos, conteo

## Evaluación (HOLD-OUT)

In [ ]:
def evaluar_modelo(embeddings, clases, k, fn_distancia, proporcion_prueba=0.2, semilla=42):
  """
  Divide aleatoriamente en entrenamiento y prueba, ejecuta KNN y retorna la exactitud (accuracy)
  """
  random.seed(semilla)
  indices = list(range(len(embeddings)))
  random.shuffle(indices)

  corte = int(len(indices) * proporcion_prueba)
  indices_prueba = indices[:corte]
  indices_train  = indices[corte:]

  emb_train  = [embeddings[i] for i in indices_train]
  cls_train  = [clases[i]     for i in indices_train]
  emb_prueba = [embeddings[i] for i in indices_prueba]
  cls_prueba = [clases[i]     for i in indices_prueba]

  aciertos = 0
  for emb, cls_real in zip(emb_prueba, cls_prueba):
      pred, _, _ = clasificar(emb, emb_train, cls_train, k, fn_distancia)
      if pred == cls_real:
          aciertos += 1

  exactitud = aciertos / len(cls_prueba) if cls_prueba else 0.0
  return exactitud

## Graficas

### Metodos de visualización

In [ ]:
def graficar_exactitud_por_k(embeddings, clases, valores_k, fn_distancia, nombre_metrica):
  """
  Genera y muestra una gráfica de exactitud vs K
  """
  exactitudes = []
  for k in valores_k:
      ex = evaluar_modelo(embeddings, clases, k, fn_distancia)
      exactitudes.append(ex)

  plt.figure(figsize=(8, 4))
  plt.plot(valores_k, [e * 100 for e in exactitudes], marker='o', color='steelblue')
  plt.title(f'Exactitud vs K  |  Métrica: {nombre_metrica}')
  plt.xlabel('Valor de K')
  plt.ylabel('Exactitud (%)')
  plt.xticks(valores_k)
  plt.grid(True, linestyle='--', alpha=0.6)
  plt.tight_layout()
  plt.show()

In [ ]:
def graficar_comparacion_metricas(embeddings, clases, k, metricas):
  """
  Gráfica de barras que compara la exactitud de cada métrica con un K fijo
  """
  nombres = []
  exactitudes = []
  for nombre, fn in metricas.values():
      ex = evaluar_modelo(embeddings, clases, k, fn)
      nombres.append(nombre)
      exactitudes.append(ex * 100)

  plt.figure(figsize=(8, 4))
  bars = plt.bar(nombres, exactitudes, color=['steelblue', 'coral', 'seagreen', 'orchid'])
  plt.title(f'Comparación de métricas  |  K = {k}')
  plt.ylabel('Exactitud (%)')
  plt.ylim(0, 100)
  for bar, val in zip(bars, exactitudes):
      plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f'{val:.1f}%', ha='center', fontsize=9)
  plt.tight_layout()
  plt.show()

### Comparacion de metricas con K=5

In [ ]:
graficar_comparacion_metricas(embeddings, clases, k=5, metricas=metricas)

### Grafica exactitud vs K para cada metrica

In [ ]:
valores_k = [1, 3, 5, 7, 9, 11]
for clave, (nombre, fn) in metricas.items():
    graficar_exactitud_por_k(embeddings, clases, valores_k, fn, nombre)

## Clasificar Entrada manual

Ejemplos de entrada para cada clase:

**`devolucion_reembolso`**
- quiero que me devuelvan el dinero de mi compra
- solicito el reembolso de mi pedido cancelado
- necesito hacer una devolución del producto que compré


**`seguimiento_pedido`**
- cuando llega mi pedido a bogota
- quiero saber el estado de mi compra
- en que va el envio de lo que compre

**`producto_danado`**
- el articulo que recibi llego roto
- el paquete estaba en mal estado cuando lo recibi
- me llegó el producto dañado qué hago

**`reclamo_entrega`**
- mi pedido lleva más de una semana y no ha llegado
- no he recibido el paquete y ya paso mucho tiempo
- sigo esperando el envio y nadie responde

**`problema_pago`**
- no me deja pagar con mi tarjeta
- el sistema rechazó mi método de pago
- tuve un problema al momento de pagar

**`consulta_producto`**
- tienen disponibilidad de ese producto en bogota
- quiero saber las características del artículo
- hay stock del producto que quiero comprar

In [ ]:
def clasificar_texto_usuario(embeddings_train, clases_train, k, fn_distancia, nombre_metrica):
  texto_usuario = input("\nIngrese el texto a clasificar: ").strip()
  embedding_usuario = modelo.encode([texto_usuario])
  embedding_usuario = list(map(float, embedding_usuario[0]))

  clase_pred, vecinos, conteo = clasificar(embedding_usuario, embeddings_train, clases_train, k, fn_distancia)

  print(f"\n--- Resultado con métrica '{nombre_metrica}' y K={k} ---")
  print(f"Clase predicha: {clase_pred}")
  print(f"Votos por clase: {conteo}")
  print(f"\nVecinos más cercanos:")
  for idx, distancia in vecinos:
      print(f"  [idx={idx}] Clase: {clases_train[idx]} | Distancia: {distancia:.4f} | Texto: {textos[idx][:60]}...")

In [ ]:
def menu_principal():
  print("\n=== CLASIFICADOR KNN ===")
  print("Métricas disponibles:")
  for clave, (nombre, _) in metricas.items():
      print(f"  {clave}. {nombre}")

  clave = input("Seleccione métrica (1-4): ").strip()
  if clave not in metricas:
      print("Opción inválida.")
      return

  nombre, fn = metricas[clave]
  k = int(input("Ingrese el valor de K: ").strip())

  # Evaluar exactitud
  exactitud = evaluar_modelo(embeddings, clases, k, fn)
  print(f"\nExactitud del modelo (hold-out 80/20): {exactitud * 100:.2f}%")

  # Clasificar texto del usuario
  clasificar_texto_usuario(embeddings, clases, k, fn, nombre)

  continuar = input("\n¿Clasificar otro texto? (s/n): ").strip().lower()
  if continuar == 's':
      menu_principal()

# Ejecutar
menu_principal()


=== CLASIFICADOR KNN ===
Métricas disponibles:
  1. Coseno
  2. Manhattan
  3. Euclidiana
  4. Minkowski
